# CVAE evaluation

Configure root.

In [ ]:
import sys, subprocess, os
import numpy as np
import pandas as pd
import torch 
import torch.nn as nn
import torch.nn.functional as F
import importlib
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/singlecell-autoencoder.git"
repo_dir = Path("singlecell-autoencoder")
if COLAB:
    root = Path("/content/singlecell-autoencoder")
    if repo_dir.exists():
        subprocess.run(["rm", "-r", root])
    subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

# Use GPU if available
generator = torch.Generator().manual_seed(111)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Data path.

In [ ]:
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = Path("/content/drive/MyDrive/phenotype-prediction-data")
    sc_path = str(data_dir / "single_cell")
    model_path = str(data_dir / "CVAE_checkpoint.pt")
    fcnts_path = str(data_dir / "fcnts_timezero")

else:
    sc_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/single_cell"
    fcnts_path = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/fcnts_timezero"

Get data.

In [ ]:
from src.sc_data import get_sc_data
from src.bulk_data import get_all_tpm_data

data = get_sc_data(path = sc_path)
bulk = get_all_tpm_data(fcnts_path = fcnts_path)

## Correlation between pseudobulk and bulk.

Plot.

In [ ]:
from src.eda import plot_pseudo_v_bulk
plot_pseudo_v_bulk(
    sc_data = data,
    bulk_data = bulk,
    timepoint = 2,
    dose = 1
)

## Comparing real v. generated cells.

Get test data.

In [ ]:
from sklearn.model_selection import train_test_split

# Train-test split
test_mask = (data["dose"] == 2) & (data["timepoint"] == 2)
train_df = data[~test_mask]
test_df = data[test_mask]
real = test_df.iloc[:, test_df.columns.str.contains("SP")].to_numpy()

Load model and sample 10000 cells.

In [ ]:
from src.vae import CVAE

checkpoint = torch.load(
    model_path,
    map_location = device,
    weights_only = True,
)

# Recreate the architecture saved during training
model = CVAE(**checkpoint["model_params"]).to(device)

# Insert the trained weights
model.load_state_dict(checkpoint["model_state_dict"])

# Set seed
seed = 111
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

model.eval()

# Sample
samples = model.generate_samples(
    num_samples = test_df.shape[0],
    t = 2, 
    d = 2
)
generated = samples.to("cpu").numpy()

Measure correlation between simulated cells and true test set cells (pseudobulk).

In [ ]:
real_mean = np.log1p(real.mean(axis=0))
generated_mean = np.log1p(generated.mean(axis=0))

pearson_r = pearsonr(real_mean, generated_mean).statistic
spearman_r = spearmanr(real_mean, generated_mean).statistic

print(f"Pseudobulk Pearson r:  {pearson_r:.4f}")
print(f"Pseudobulk Spearman r: {spearman_r:.4f}")

plt.figure(figsize=(6, 6))
plt.scatter(real_mean, generated_mean, s=8, alpha=0.4)

limits = [
    min(real_mean.min(), generated_mean.min()),
    max(real_mean.max(), generated_mean.max()),
]
plt.plot(limits, limits, "k--", linewidth=1)

plt.xlabel("Real mean log1p(CPM)")
plt.ylabel("Generated mean log1p(CPM)")
plt.title(f"Pseudobulk expression: Pearson r = {pearson_r:.3f}")
plt.tight_layout()
plt.show()

## Latent space clustering


Encode all cells to check if latent embeddings are condition agnostic.

In [ ]:
from src.eval import plot_latent_pca

plot_latent_pca(
    data = data,
    model = model,
    color_by = "dose"
)

Examine condition-embedded cells to see how cells spearate in latent space.

In [ ]:
from src.eval import plot_composed_latent_pca

timepoints = [0, 1, 2, 3, 4, 5]
doses = [2]
num_samples = 1000
color_by = "timepoint"

plot_composed_latent_pca(
    model = model,
    timepoints = timepoints,
    doses = doses,
    num_samples = num_samples,
    color_by = color_by,
    title = "PCA on composed latents generated for 2x MIC"
)